## Use requests to get data

In [32]:
import requests # HTTP library, used to call the openFDA API

In [ ]:
# 调用 openFDA 药品标签接口，limit=5 表示只取 5 份标签
drug = requests.get("https://api.fda.gov/drug/label.json", params={"limit": 5})

drug.raise_for_status()  # 如果接口返回错误（如 404、429 限流），在这里直接报错，而不是后面出现难懂的 KeyError

In [34]:
data = drug.json()  # 把返回的 JSON 文本解析成 Python 字典

labels = data["results"]  # 标签列表在 "results" 键下，每个元素是一份标签（dict）

In [35]:
labels[0].keys()  # 查看第一份标签有哪些字段

dict_keys(['effective_time', 'inactive_ingredient', 'purpose', 'keep_out_of_reach_of_children', 'warnings', 'questions', 'spl_product_data_elements', 'openfda', 'version', 'dosage_and_administration', 'pregnancy_or_breast_feeding', 'stop_use', 'storage_and_handling', 'do_not_use', 'package_label_principal_display_panel', 'indications_and_usage', 'set_id', 'id', 'active_ingredient'])

In [36]:
labels[0]["indications_and_usage"]  # 查看适应症字段的内容（是一个字符串列表）

['INDICATIONS Condition listed above or as directed by the physician']

In [37]:
#labels[0]["openfda"]["generic_name"][0]

In [38]:
# how to deal with the case when the key doesn't exist in the dictionary? Use .get(key, default) to avoid KeyError.
# do not assume that the key exists, length of the list, or has fallback values.

def get_drug_name(label: dict) -> str:
    """
    Extracts the drug name from the label dictionary.
    
    Args:
        label (dict): A dictionary containing drug label information.

    Returns:
        str: The drug name.
    """
    # .get(key, default) is used to avoid KeyError if the key doesn't exist
    openfda = label.get("openfda", {})  # 取 openfda 子字典，没有就用空字典
    # 按优先级找名字：通用名 → 品牌名 → 成分名；`or` 会返回第一个非空的值，都为空时返回 []
    names = openfda.get("generic_name") or openfda.get("brand_name") or openfda.get("substance_name") or []
    return ", ".join(names) if names else "Unknown Drug Name"  # 多个名字用逗号连接；一个都没有就返回占位文字

In [39]:
# validate the function by printing the drug names for each label
for i, l in enumerate(labels):
    print(i, l.get("openfda", {}).get("generic_name", "【缺失】"))

0 ['SILICEA']
1 【缺失】
2 ['POVIDONE-IODINE']
3 【缺失】
4 【缺失】


In [40]:
for i, l in enumerate(labels):
    print(i, get_drug_name(l))

0 SILICEA
1 Unknown Drug Name
2 POVIDONE-IODINE
3 Unknown Drug Name
4 Unknown Drug Name


In [41]:
labels[1].get("openfda")

{}

## Define a Schema using Pydantic

In [42]:
from pydantic import BaseModel, Field  # BaseModel 定义数据结构；Field 给字段加默认值和描述
from typing import Literal, Optional   # Literal 限定只能取几个固定值；Optional[X] 表示 X 或 None

# 把白名单内的标签段落合并成一段文本（字段列表见 label_to_text.py 里的 FIELDS）
import importlib, label_to_text as _l2t  # importlib 用来重新加载模块
importlib.reload(_l2t)  # 修改 label_to_text.py 后不用重启 kernel 也能生效（Python 默认会缓存已导入的模块）
from label_to_text import label_to_text  # 从重新加载后的模块里导入函数

In [43]:
# Schema 的结构对应 FDA 标签里安全信息的分级，
# 让模型每类信息都有地方放，而不是全部塞进一个列表

# 允许的剂型：模型只能从这里选，避免出现 "Pellets" / "Bark" / "ophthalmic solution" 这种五花八门的写法
DosageForm = Literal[
    "tablet", "capsule", "pellet", "powder", "liquid", "solution", "suspension",  # 片剂、胶囊、小丸、粉剂、液体、溶液、混悬液
    "syrup", "cream", "ointment", "gel", "lotion", "spray", "drops",             # 糖浆、乳膏、软膏、凝胶、洗剂、喷雾、滴剂
    "injection", "patch", "suppository", "other",                                # 注射剂、贴剂、栓剂、其他
]

class SafetyInfo(BaseModel):  # 安全信息，分三级
    contraindications: list[str] = Field(  # 第一级：绝对禁忌
        default_factory=list,  # 默认是空列表（用 default_factory 避免多个实例共用同一个列表）
        description="Absolute prohibitions: conditions under which the drug must never be used "  # 这段描述会发给模型，告诉它该放什么
        "(from CONTRAINDICATIONS on Rx labels or DO_NOT_USE on OTC labels), e.g. known allergy "  # 相邻字符串会自动拼成一个
        "to an ingredient. Empty list if the label has none.",
    )
    cautions: list[str] = Field(  # 第二级：慎用，需要咨询医生
        default_factory=list,  # 默认空列表
        description="Conditions requiring care or a doctor's advice before use "  # 给模型的说明
        "(ASK_DOCTOR, WARNINGS), e.g. pregnancy or breast-feeding, liver disease.",
    )
    stop_use_conditions: list[str] = Field(  # 第三级：出现某情况时停药
        default_factory=list,  # 默认空列表
        description="Situations in which the user should stop taking the drug "  # 给模型的说明
        "(STOP_USE), e.g. symptoms persist more than N days.",
    )

class DrugLabel(BaseModel):  # 模型最终要返回的完整结构
    indications: list[str] = Field(  # 适应症
        default_factory=list,  # 默认空列表
        description=(  # 给模型一条明确的决策规则，消除“这句算不算适应症”的歧义
            "Specific medical conditions or symptoms this product treats or relieves. "  # 要什么：具体的病症或症状
            "Look in INDICATIONS_AND_USAGE and PURPOSE sections. "  # 去哪找：告诉模型看哪两个段落
            "Exclude non-informative phrases such as 'condition listed above' or "  # 不要什么：没有信息量的套话
            "'as directed by the physician'. "
            "Each item must be an independently meaningful condition "  # 每一项单独拿出来也要有意义
            "(e.g. 'acne', not 'temporary relief')."  # 正反例各一个，比抽象规则更好用
        ),
    )
    safety: SafetyInfo = Field(  # 嵌套上面定义的 SafetyInfo
        description="Safety statements about the patient. Never include packaging, tamper "  # 明确排除包装、封口、储存等非人体相关信息
        "seals, storage, 'external use only', or 'keep out of reach of children'.",
    )
    dosage_form: Optional[DosageForm] = Field(  # 剂型：只能是 DosageForm 里的值，或 None
        default=None,  # 标签没写时为 None
        description="Physical form of the product as administered, taken from dosage or "  # 让模型从用法里判断剂型
        "administration text (e.g. 'dissolve under the tongue' pellets => pellet). "
        "Not the plant part or raw ingredient. Null if not stated.",  # 防止把原料部位（如 BARK 树皮）当成剂型
    )

## OpenAI API

In [44]:
from openai import OpenAI  # OpenAI 官方 SDK，用来调用模型
from dotenv import load_dotenv  # 从 .env 文件读取环境变量（例如 OPENAI_API_KEY）

In [45]:
load_dotenv()  # 把 .env 里的变量加载到环境变量中

client = OpenAI()  # 创建客户端；会自动从环境变量读取 OPENAI_API_KEY

# 系统提示词：告诉模型它的角色，以及只能使用原文里的信息（减少编造）
SYSTEM_PROMPT = "You extract structured information from FDA drug labels. Only use information present in the text."

# 没有信息量的套话：包含这些短语的 indication 会被代码直接删掉（全部用小写，方便比较）
NON_INFORMATIVE = {"condition listed above", "as directed by the physician", "as directed"}

def clean_indications(items: list[str]) -> list[str]:
    """后处理：删掉套话，并去掉重复项（忽略大小写和首尾空格）。"""
    cleaned = []  # 保存保留下来的条目
    seen = set()  # 记录已经出现过的条目（小写形式），用来去重
    for item in items:  # 逐条检查模型给出的 indication
        key = item.strip().lower()  # 统一成小写、去掉首尾空格，便于比较
        if not key:  # 空字符串
            continue  # 跳过
        if any(p in key for p in NON_INFORMATIVE):  # 只要包含任意一个套话短语
            continue  # 就丢掉这一条
        if key in seen:  # 之前已经出现过（例如 "Acne" 和 "acne"）
            continue  # 丢掉重复项
        seen.add(key)  # 记录下来
        cleaned.append(item.strip())  # 保留原始大小写，只去掉首尾空格
    return cleaned  # 返回清洗后的列表

def extract_drug_label_info(label_text: str, clean: bool = True) -> DrugLabel | None:
    """
    Extract structured information from a drug label text using OpenAI API.
    clean=True 时对 indications 做规则过滤；clean=False 返回模型原始结果（用来对比过滤的效果）。
    Returns None if the model refuses or its output can't be parsed into DrugLabel.
    """
    response = client.responses.parse(  # Responses API + 结构化输出：SDK 会把 DrugLabel 转成 JSON Schema 发给模型
        model="gpt-4o-mini",  # 使用的模型
        temperature=0,  # 降低随机性；但它解决不了“两种答案本来就一样合理”的歧义，歧义要在输入侧解决
        input=[  # 对话消息列表
            {"role": "system", "content": SYSTEM_PROMPT},  # 系统消息：规则
            {"role": "user", "content": label_text}  # 用户消息：标签文本
        ],
        text_format=DrugLabel,  # 要求模型按 DrugLabel 的结构输出
    )

    # Responses API 没有 .choices（那是 Chat Completions API 的写法）
    # output_parsed 已经是 DrugLabel 实例；output_text 是模型返回的原始 JSON 字符串
    result = response.output_parsed  # 取出解析好的结果
    if result is None:  # 模型拒绝回答或输出不符合 schema 时为 None
        print("Could not parse model output:", response.output_text)  # 打印原始输出，方便排查
        return None  # 直接返回 None
    if clean:  # 需要清洗时
        result.indications = clean_indications(result.indications)  # 用规则过滤 indications
    return result  # 返回 DrugLabel 实例

In [46]:
for i, label in enumerate(labels):  # enumerate 同时给出序号 i 和标签 label
    label_text = label_to_text(label)  # 把白名单字段合并成一段文本
    print(f"===== label {i}: {get_drug_name(label)} ({len(label_text)} chars) =====")  # 打印序号、药名和 prompt 长度（字符数，可粗略反映成本）
    result = extract_drug_label_info(label_text)  # 调用模型抽取结构化信息
    if result is None:  # 解析失败
        continue  # 跳过这份标签，继续下一份
    print(result.model_dump_json(indent=2))  # 把结果转成带缩进的 JSON 打印
    print("original label text:", label_text, "\n")  # 打印原文，方便人工对照检查

===== label 0: SILICEA (1055 chars) =====
{
  "indications": [
    "Acne",
    "Boils"
  ],
  "safety": {
    "contraindications": [
      "Do not use if capseal is broken or missing."
    ],
    "cautions": [
      "If pregnant or nursing, seek professional advice before taking this product."
    ],
    "stop_use_conditions": [
      "If symptoms do not improve in 4 days, or worsen, discontinue use and seek assistance of health professional."
    ]
  },
  "dosage_form": "pellet"
}
original label text: INDICATIONS_AND_USAGE:
INDICATIONS Condition listed above or as directed by the physician

PURPOSE:
USES USES: Temporary Relief - Acne, Boils* * Claims based on traditional homeopathic practice, not accepted medical evidence. Not FDA evaluated.

DO_NOT_USE:
Do not use if capseal is broken or missing. Close the cap tightly after use.

WARNINGS:
WARNINGS This product is to be used for self-limiting conditions If symptoms do not improve in 4 days, or worsen, discontinue use and seek assista

In [49]:
for run in range(3):
    result = extract_drug_label_info(label_to_text(labels[0]))
    print(f"--- run {run} ---")
    print(result.model_dump_json(indent=2))

--- run 0 ---
{
  "indications": [
    "Acne",
    "Boils"
  ],
  "safety": {
    "contraindications": [
      "Do not use if capseal is broken or missing."
    ],
    "cautions": [
      "If pregnant or nursing, seek professional advice before use."
    ],
    "stop_use_conditions": [
      "If symptoms do not improve in 4 days, or worsen."
    ]
  },
  "dosage_form": "pellet"
}
--- run 1 ---
{
  "indications": [
    "Acne",
    "Boils"
  ],
  "safety": {
    "contraindications": [
      "Do not use if capseal is broken or missing."
    ],
    "cautions": [
      "If pregnant or nursing, seek professional advice before use."
    ],
    "stop_use_conditions": [
      "If symptoms do not improve in 4 days, or worsen."
    ]
  },
  "dosage_form": "pellet"
}
--- run 2 ---
{
  "indications": [
    "acne",
    "boils"
  ],
  "safety": {
    "contraindications": [
      "Do not use if capseal is broken or missing."
    ],
    "cautions": [
      "If pregnant or nursing, seek professional adv

In [48]:
# 一致性测试：每份标签跑 3 次，检查 indications 是否稳定
# 同时记录“过滤前”（模型原始输出）和“过滤后”，区分稳定性来自 description 规则还是代码过滤
N_RUNS = 3  # 每份标签跑几次

def as_set(items: list[str]) -> frozenset:
    """把列表转成忽略大小写、忽略顺序的集合，只比较内容。"""
    return frozenset(i.strip().lower() for i in items)  # frozenset 可以放进 set 里做比较

for i, label in enumerate(labels):  # 遍历 5 份标签
    label_text = label_to_text(label)  # 生成 prompt 文本
    raw_runs, clean_runs = [], []  # 分别保存每次的原始结果和过滤后结果
    for run in range(N_RUNS):  # 跑 N_RUNS 次
        result = extract_drug_label_info(label_text, clean=False)  # 取原始结果（不过滤）
        if result is None:  # 解析失败
            continue  # 这次不计入
        raw_runs.append(result.indications)  # 保存原始 indications
        clean_runs.append(clean_indications(result.indications))  # 用同一次结果做过滤，保证对比公平

    raw_stable = len({as_set(r) for r in raw_runs}) == 1  # 所有次的集合都相同 → 稳定
    clean_stable = len({as_set(r) for r in clean_runs}) == 1  # 过滤后是否稳定
    print(f"===== label {i}: {get_drug_name(label)} =====")  # 标题
    print(f"raw stable: {raw_stable} | cleaned stable: {clean_stable}")  # 一眼看出结论
    for run, (raw, cleaned) in enumerate(zip(raw_runs, clean_runs), start=1):  # 逐次打印细节
        print(f"  run {run} raw    : {raw}")  # 模型原始输出
        print(f"  run {run} cleaned: {cleaned}")  # 过滤后的结果
    print()  # 空一行，分隔不同标签

===== label 0: SILICEA =====
raw stable: True | cleaned stable: True
  run 1 raw    : ['Acne', 'Boils']
  run 1 cleaned: ['Acne', 'Boils']
  run 2 raw    : ['Acne', 'Boils']
  run 2 cleaned: ['Acne', 'Boils']
  run 3 raw    : ['acne', 'boils']
  run 3 cleaned: ['acne', 'boils']

===== label 1: Unknown Drug Name =====
raw stable: True | cleaned stable: True
  run 1 raw    : ['broad-spectrum SPF 46 protection', 'flawless skin']
  run 1 cleaned: ['broad-spectrum SPF 46 protection', 'flawless skin']
  run 2 raw    : ['broad-spectrum SPF 46 protection', 'flawless skin']
  run 2 cleaned: ['broad-spectrum SPF 46 protection', 'flawless skin']
  run 3 raw    : ['broad-spectrum SPF 46 protection', 'flawless skin']
  run 3 cleaned: ['broad-spectrum SPF 46 protection', 'flawless skin']

===== label 2: POVIDONE-IODINE =====
raw stable: True | cleaned stable: True
  run 1 raw    : ['first aid to help prevent infection in minor cuts', 'scrapes', 'burns']
  run 1 cleaned: ['first aid to help prevent i

In [50]:
import json                                  # 标准库：读写 JSON
from datetime import datetime, timezone      # 记录运行时间（UTC）
from pathlib import Path                     # 跨平台的路径处理

# 找项目根目录：从当前目录往上找，第一个包含 pyproject.toml 的目录就是根目录
# （notebook 在 src/ 里运行，直接写 "data/samples.json" 会存到 src/data/ 下，位置不对）
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
OUT_PATH = ROOT / "data" / "samples.json"    # 输出文件：<项目根>/data/samples.json
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)  # data/ 不存在就创建；已存在不报错

samples = []                                 # 用来收集每份标签的记录
for i, label in enumerate(labels):           # 遍历 5 份标签，i 是序号
    label_text = label_to_text(label)        # 输入：真正发给模型的文本（白名单字段）
    result = extract_drug_label_info(label_text)  # 输出：调用模型（已含 temperature=0 和过滤）
    samples.append({                         # 每份标签存成一个 dict
        "index": i,                          # 在本批数据里的序号
        "id": label.get("id"),               # openFDA 的标签版本 ID，以后可以用它重新拉取原始数据
        "set_id": label.get("set_id"),       # 同一产品所有版本共用的 ID
        "drug_name": get_drug_name(label),   # 药名（来自 openfda 字段，不经过模型）
        "input_text": label_text,            # 模型的输入
        "input_chars": len(label_text),      # 输入长度，用来粗略比较成本
        # 模型的输出：model_dump() 把 pydantic 对象转成普通 dict；解析失败时存 None（JSON 里是 null）
        "output": result.model_dump() if result is not None else None,
    })

payload = {                                  # 文件的顶层结构：元数据 + 样本列表
    "created_at": datetime.now(timezone.utc).isoformat(),  # 运行时间，方便区分不同版本的结果
    "model": "gpt-4o-mini",                  # 用的模型（改模型时记得同步改这里）
    "temperature": 0,                        # 采样参数
    "system_prompt": SYSTEM_PROMPT,          # 用的系统提示词
    "fields": list(_l2t.FIELDS),             # 用的字段白名单，结果不同时可以追溯输入差异
    "schema": DrugLabel.model_json_schema(), # 当时的 schema（含 description），prompt 改动也能追溯
    "samples": samples,                      # 5 份标签的输入输出
}

with open(OUT_PATH, "w", encoding="utf-8") as f:  # 用 utf-8 打开，避免特殊字符出错
    json.dump(payload, f, indent=2, ensure_ascii=False)  # indent=2 方便阅读；ensure_ascii=False 让非英文字符原样保存

print(f"saved {len(samples)} samples to {OUT_PATH}")  # 确认保存成功和保存位置

saved 5 samples to /Users/valerieyin/GitRepo/drug-label-extractor/data/samples.json
